# ZF / 美债期货期权链调试 notebook

这个 notebook 用 `ib_async` 逐步调试：连接 IB Gateway、枚举多个底层期货月份的 FOP 期权链、分批 qualify、并发订阅报价 / OI / 成交量 / Greeks。

关键点：不要只查 `ZF2606` 一个底层 conId。临近到期时，`ZF2606` 下面可能只剩少量期权到期日，后续日期通常挂在 `ZF2609`、`ZF2612` 等后续底层期货合约上。


In [ ]:
from ib_async import IB, util
import pandas as pd
from pathlib import Path
import random

from treasury_fop_chain import (
    attach_error_collector,
    detach_error_collector,
    discover_future_months,
    qualify_future,
    get_future_price,
    get_future_prices_for_months,
    get_reference_price,
    build_treasury_fop_universe,
    save_universe,
    load_universe,
    FOPMarketDataStreamer,
    snapshot_in_batches,
    select_atm_contracts,
    filter_contracts_by_moneyness,
    prepare_snapshot_features,
    summarize_snapshot,
)

util.startLoop()
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 240)


## 1. 参数

`FUTURE_MONTHS` 是最重要的参数。比如当前要完整覆盖 ZF 期权，不要只填 `202606`，而是把后续活跃底层月份一起填上。

In [ ]:
ROOT = "ZF"
FUTURE_MONTHS = ["202606", "202609", "202612"]

HOST = "127.0.0.1"
PORT = 4002          # IB Gateway paper: 4002; live: 4001; TWS paper: 7497
CLIENT_ID = random.randint(2000, 9999)
MARKET_DATA_TYPE = 1 # 1=live, 2=frozen, 3=delayed, 4=delayed frozen

UNIVERSE_CSV = Path(f"{ROOT}_FOP_Universe_{'_'.join(FUTURE_MONTHS)}.csv")
SNAPSHOT_CSV = Path(f"{ROOT}_FOP_Snapshot.csv")

ROOT, FUTURE_MONTHS, CLIENT_ID, UNIVERSE_CSV


## 2. 连接 IB Gateway

如果看到 `10197`，通常是其他终端正在抢实时行情权限；如果看到 `354`，通常是没有对应市场数据订阅。

In [ ]:
ib = IB()
ib.connect(HOST, PORT, clientId=CLIENT_ID, timeout=10)
ib.reqMarketDataType(MARKET_DATA_TYPE)
errors, error_handler = attach_error_collector(ib)

print("connected:", ib.isConnected(), "clientId:", CLIENT_ID)


## 3. Get current future price

Use this cell to fetch the current price of a specified Treasury futures contract.

In [ ]:
price_info = get_future_price(
    ib,
    ROOT,
    FUTURE_MONTHS[0],
    market_data_type=MARKET_DATA_TYPE,
)

print("contract:", price_info["localSymbol"], "conId:", price_info["conId"])
print("price:", price_info["price"], "source:", price_info["priceSource"])
pd.Series({k: v for k, v in price_info.items() if k not in ["contract", "ticker"]})


## 3. 可选：自动发现后续底层期货月份

如果你不确定应该填哪些月份，先跑这个单元看 IB 返回了哪些 FUT 合约月份。

In [ ]:
# 可选：如果 FUTURE_MONTHS 不确定，打开下面两行。
discovered = discover_future_months(ib, ROOT, min_month="202606", max_count=10)
discovered


## 4. 枚举并缓存全量期权链

这里会对每一个底层期货月份分别调用 `reqSecDefOptParams(root, 'CBOT', 'FUT', underlying.conId)`，再按返回的 `tradingClass / expirations / strikes` 构造 FOP 合约。分批 `qualifyContracts` 后会保存 conId，后续刷新行情可以直接从 CSV 加载，不必每次重新 qualify。

In [ ]:
universe = build_treasury_fop_universe(
    ib,
    root=ROOT,
    future_months=FUTURE_MONTHS,
    exchange="CBOT",
    fop_exchange="CBOT",
    qualify_batch_size=200,
)

save_universe(universe, UNIVERSE_CSV)
print("valid FOP contracts:", len(universe.contracts))
print("saved:", UNIVERSE_CSV)
display(universe.chain_summary)
display(universe.metadata.head(20))


## 5. 从缓存加载 universe

调试 market data 时建议从这里开始反复跑，减少 IB contract details 请求。

In [ ]:
contracts, universe_df = load_universe(UNIVERSE_CSV)
universe_df["expiration"] = universe_df["lastTradeDateOrContractMonth"].astype(str)

summary = universe_df.groupby("underlyingMonth", dropna=False).agg(
    contracts=("conId", "count"),
    expirations=("expiration", "nunique"),
    firstExpiration=("expiration", "min"),
    lastExpiration=("expiration", "max"),
    minStrike=("strike", "min"),
    maxStrike=("strike", "max"),
)

print("loaded contracts:", len(contracts))
display(summary)
display(universe_df)


## 6. 过滤需要监控的合约

规则：0DTE 合约只保留距离当前期货价格 2 点以内的 strike；非 0DTE 合约保留 5 点以内。不同底层月份会分别使用对应期货合约的价格。

In [ ]:
future_prices = get_future_prices_for_months(
    ib,
    ROOT,
    FUTURE_MONTHS,
    market_data_type=MARKET_DATA_TYPE,
)
spot_by_month = dict(zip(future_prices["month"].astype(str), future_prices["price"].astype(float)))

filtered_contracts, filtered_universe_df = filter_contracts_by_moneyness(
    contracts,
    universe_df,
    spot_by_underlying_month=spot_by_month,
    dte0_width=2.0,
    non_dte0_width=5.0,
)

print("original contracts:", len(contracts))
print("filtered contracts:", len(filtered_contracts))
display(future_prices)
display(
    filtered_universe_df.groupby(["underlyingMonth", "expiration", "dte0"], dropna=False)
    .agg(contracts=("conId", "count"), minStrike=("strike", "min"), maxStrike=("strike", "max"))
    .head(30)
)


## 6. 小样本测试报价 / Greeks

先用 80 个合约确认权限、字段和到达速度。`genericTickList='100,101,106'` 会请求期权成交量、OI、IV/Greeks。

In [ ]:
# 不要用 contracts[:80] 做样本：排序最前面通常全是当日到期合约，IB 很容易返回 Model is not valid。
# 这里改为：优先从过滤后的 universe 中选非当日到期、ATM 附近、最近几个到期日。
today = pd.Timestamp.now(tz="Asia/Shanghai").strftime("%Y%m%d")
front_future = qualify_future(ib, ROOT, FUTURE_MONTHS[0])
fallback_spot = float(pd.to_numeric(universe_df["strike"], errors="coerce").median())
spot = get_reference_price(ib, front_future, fallback=fallback_spot)

sample_source_contracts = filtered_contracts if "filtered_contracts" in globals() else contracts
sample_source_df = filtered_universe_df if "filtered_universe_df" in globals() else universe_df

sample_contracts, sample_meta = select_atm_contracts(
    sample_source_contracts,
    sample_source_df,
    spot=spot,
    expiration_after=today,
    underlying_month=FUTURE_MONTHS[0],
    max_expirations=3,
    strikes_each_side=5,
)

print("today:", today, "spot:", spot, "sample size:", len(sample_contracts))
display(sample_meta[["underlyingMonth", "localSymbol", "expiration", "right", "strike", "conId"]].head(30))

sample_streamer = FOPMarketDataStreamer(ib, request_interval=0.025)
sample_streamer.subscribe(sample_contracts)
sample_stats = sample_streamer.wait_until_stable(min_seconds=5, max_seconds=20, stable_seconds=3)
sample_df = sample_streamer.snapshot()
sample_streamer.cancel()
sample_features = prepare_snapshot_features(sample_df, include_chinese_columns=True)

display(sample_stats)
display(summarize_snapshot(sample_df))
display(sample_features.head(20))

## 7. 全量并发订阅：最快刷新方式

这一步会保持所有 market data line 打开。之后每次 `streamer.snapshot()` 都只是读取内存中已经被 IB event loop 更新的 Ticker，速度最快。

如果这里报 market data line 上限，改用下一节的分批模式，或把 `active_contracts = contracts[:300]` 改小。

In [ ]:
active_contracts = filtered_contracts if "filtered_contracts" in globals() else contracts
print("subscribing contracts:", len(active_contracts))

streamer = FOPMarketDataStreamer(ib, request_interval=0.025)
streamer.subscribe(active_contracts)
stats = streamer.wait_until_stable(max_seconds=20, stable_seconds=2)

df = streamer.snapshot()
df_features = prepare_snapshot_features(df, include_chinese_columns=True)
df.to_csv(SNAPSHOT_CSV, index=False, encoding="utf-8-sig")
FEATURES_CSV = SNAPSHOT_CSV.with_name(SNAPSHOT_CSV.stem + "_features_cn.csv")
df_features.to_csv(FEATURES_CSV, index=False, encoding="utf-8-sig")

display(stats)
display(summarize_snapshot(df))
display(df_features.head(20))
print("raw saved:", SNAPSHOT_CSV)
print("features saved:", FEATURES_CSV)

## 8. 快速刷新快照

保持上一节 `streamer` 不 cancel，然后反复运行这个单元即可刷新整条链的最新报价和 Greeks。

In [ ]:
ib.sleep(2)
df_refresh = streamer.snapshot()
df_refresh_features = prepare_snapshot_features(df_refresh, include_chinese_columns=True)
df_refresh.to_csv(SNAPSHOT_CSV, index=False, encoding="utf-8-sig")
FEATURES_CSV = SNAPSHOT_CSV.with_name(SNAPSHOT_CSV.stem + "_features_cn.csv")
df_refresh_features.to_csv(FEATURES_CSV, index=False, encoding="utf-8-sig")

display(summarize_snapshot(df_refresh))
display(df_refresh_features.head(20))
print("features saved:", FEATURES_CSV)

## 9. 备选：分批并发快照

如果账户行情线数量不足，无法全量保持订阅，就用这个模式。它比保持订阅慢，但每批内部仍然是并发 market data streams。

In [ ]:
# 如需使用分批模式，先取消上一节的 streamer。
# streamer.cancel()
#
# batch_contracts = filtered_contracts if "filtered_contracts" in globals() else contracts
# df_batch = snapshot_in_batches(
#     ib,
#     batch_contracts,
#     batch_size=300,
#     wait_max_seconds=15,
#     wait_stable_seconds=2,
# )
# df_batch_features = prepare_snapshot_features(df_batch, include_chinese_columns=True)
# df_batch.to_csv(SNAPSHOT_CSV, index=False, encoding="utf-8-sig")
# FEATURES_CSV = SNAPSHOT_CSV.with_name(SNAPSHOT_CSV.stem + "_features_cn.csv")
# df_batch_features.to_csv(FEATURES_CSV, index=False, encoding="utf-8-sig")
# display(summarize_snapshot(df_batch))
# display(df_batch_features.head(20))

## 10. 查看 IB API 错误

重点看 `10197`、`354`、`420`、`100/101`。这些通常分别对应其他终端抢权限、无行情权限、节流/限速、消息速率或行情线限制。

In [ ]:
pd.DataFrame(errors).tail(30)


## 11. 清理连接

跑完一定取消 market data，不然 IB Gateway 会继续占用行情线。

In [ ]:
for name in ["streamer", "sample_streamer"]:
    obj = globals().get(name)
    if obj is not None:
        obj.cancel()

if "error_handler" in globals():
    detach_error_collector(ib, error_handler)

if ib.isConnected():
    ib.disconnect()

print("disconnected")
